# Exercice 2 — Choisir et créer des visualisations pertinentes — CORRECTION

**Difficulté :** Intermédiaire  
**Durée estimée :** 40 min  

---

## Contexte

Le directeur de production souhaite comprendre les tendances de la ligne de fabrication. On dispose du fichier `production_industrielle.csv` qui contient les relevés journaliers de production pour trois lignes.

La donnée brute ne parle pas d'elle-même : c'est le graphique approprié qui révèle les patterns. Le choix du type de graphique n'est pas anodin — un mauvais choix peut masquer l'information importante ou induire en erreur.

**Objectif :** Charger, nettoyer et visualiser les données de production en produisant 3 graphiques pertinents avec plotly, chacun accompagné d'une interprétation.

---

## Données

Fichier : `../data/production_industrielle.csv`

---

## Tâche 1 — Charger et nettoyer les données

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Chargement
df = pd.read_csv('../data/production_industrielle.csv')

print("Dimensions initiales :", df.shape)
print()
print("Aperçu :")
df.head()

In [ ]:
# Diagnostic des données
print("Valeurs manquantes :")
print(df.isnull().sum())
print()
print("Défauts négatifs (erreurs de saisie) :", (df['quantite_defauts'] < 0).sum())

In [ ]:
# Nettoyage 1 : conversion de la date
df['date'] = pd.to_datetime(df['date'])

# Nettoyage 2 : remplacer les valeurs NaN de temps_arret_min par 0
# Hypothèse métier : pas de données = pas d'arrêt enregistré
df['temps_arret_min'] = df['temps_arret_min'].fillna(0)

# Nettoyage 3 : supprimer les lignes avec des défauts négatifs (valeurs impossibles)
df = df[df['quantite_defauts'] >= 0]

print("Dimensions après nettoyage :", df.shape)
print("Valeurs manquantes restantes :")
print(df.isnull().sum())

**Explication :** Le nettoyage précède toujours la visualisation. Trois décisions ont été prises ici, chacune avec une justification métier :

1. **Conversion datetime** : obligatoire pour tout axe temporel et tout calcul de rolling window
2. **NaN → 0 pour les arrêts** : un arrêt non renseigné est vraisemblablement un arrêt absent (durée nulle), pas une donnée inconnue
3. **Suppression des défauts négatifs** : une quantité de défauts ne peut pas être négative — ces lignes sont des erreurs de saisie à écarter

Dans un contexte réel, ces décisions seraient validées avec le responsable qualité avant d'être appliquées.

## Tâche 2 — Graphique 1 : production totale par ligne

**Choix du type :** Bar chart. Les barres permettent de comparer des valeurs discrètes entre catégories (ici : les trois lignes de production). La hauteur de chaque barre encode directement la quantité, ce qui facilite la lecture comparative. Un pie chart aurait été moins précis pour des valeurs proches.

In [ ]:
# Agréger la production totale par ligne
prod_par_ligne = df.groupby('ligne')['quantite_produite'].sum().reset_index()
prod_par_ligne.columns = ['ligne', 'production_totale']
prod_par_ligne = prod_par_ligne.sort_values('production_totale', ascending=False)

# Créer le bar chart
fig1 = px.bar(
    prod_par_ligne,
    x='ligne',
    y='production_totale',
    color='ligne',                      # une couleur par barre
    title='Production totale par ligne de production',
    labels={
        'ligne': 'Ligne de production',
        'production_totale': 'Quantité produite (unités)'
    },
    text='production_totale',           # afficher la valeur sur chaque barre
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig1.update_traces(texttemplate='%{text:,}', textposition='outside')
fig1.update_layout(
    showlegend=False,
    yaxis_title='Quantité produite (unités)',
    xaxis_title='Ligne de production',
    plot_bgcolor='white'
)

fig1.show()

**Interprétation :** Les trois lignes de production affichent des volumes relativement équilibrés sur la période. Si une ligne se détache significativement, cela peut indiquer une capacité nominale différente, une cadence réduite (maintenance), ou un mix produit défavorable. Ce graphique donne le point de départ pour investiguer les écarts.

## Tâche 3 — Graphique 2 : évolution quotidienne avec moyenne mobile

**Choix du type :** Line chart. Les données sont continues dans le temps et ordonnées chronologiquement — la courbe est le seul type de graphique qui encode correctement cette continuité. La moyenne mobile lisse les variations journalières pour révéler la tendance de fond.

In [ ]:
# Agréger la production journalière toutes lignes confondues
prod_journaliere = df.groupby('date')['quantite_produite'].sum().reset_index()
prod_journaliere.columns = ['date', 'production_totale']
prod_journaliere = prod_journaliere.sort_values('date')

# Calculer la moyenne mobile sur 7 jours
# min_periods=1 : calculer même quand il y a moins de 7 jours disponibles (début de série)
prod_journaliere['moyenne_mobile_7j'] = (
    prod_journaliere['production_totale']
    .rolling(window=7, min_periods=1)
    .mean()
    .round(0)
)

# Construire le graphique avec deux traces
fig2 = go.Figure()

# Trace 1 : production réelle (fond discret, peu opaque)
fig2.add_trace(go.Scatter(
    x=prod_journaliere['date'],
    y=prod_journaliere['production_totale'],
    mode='lines',
    name='Production journalière',
    line=dict(color='steelblue', width=1),
    opacity=0.5
))

# Trace 2 : moyenne mobile (mise en avant, ligne épaisse)
fig2.add_trace(go.Scatter(
    x=prod_journaliere['date'],
    y=prod_journaliere['moyenne_mobile_7j'],
    mode='lines',
    name='Moyenne mobile 7 jours',
    line=dict(color='darkorange', width=2.5)
))

fig2.update_layout(
    title='Évolution quotidienne de la production (avec tendance 7 jours)',
    xaxis_title='Date',
    yaxis_title='Quantité produite (unités)',
    plot_bgcolor='white',
    legend=dict(x=0.01, y=0.99)
)

fig2.show()

**Interprétation :** La courbe journalière montre une volatilité naturelle (weekends, arrêts ponctuels), tandis que la moyenne mobile sur 7 jours révèle la tendance réelle. Une tendance décroissante sur la moyenne mobile signalerait un problème structurel à investiguer. La fenêtre de 7 jours est pertinente ici car elle correspond à un cycle hebdomadaire complet.

**Explication technique :** `.rolling(window=7)` crée une fenêtre glissante de 7 observations. Pour chaque point, pandas calcule la moyenne des 7 valeurs précédentes. `min_periods=1` évite les NaN en début de série.

## Tâche 4 — Graphique 3 : quantité produite vs défauts par équipe

**Choix du type :** Scatter plot. L'objectif est d'explorer la relation entre deux variables numériques continues — la corrélation éventuelle entre volume produit et nombre de défauts. Chaque point représente une session de travail. La coloration par équipe permet de vérifier si le comportement diffère selon l'équipe.

In [ ]:
fig3 = px.scatter(
    df,
    x='quantite_produite',
    y='quantite_defauts',
    color='operateur',                     # couleur par équipe
    title='Relation entre production et défauts, par équipe',
    labels={
        'quantite_produite': 'Quantité produite (unités)',
        'quantite_defauts': 'Quantité de défauts (unités)',
        'operateur': 'Équipe'
    },
    opacity=0.6,                           # transparence pour voir les superpositions
    trendline='ols',                       # droite de régression par groupe
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig3.update_layout(
    plot_bgcolor='white',
    legend_title_text='Équipe'
)

fig3.show()

**Interprétation :** Si une droite de tendance positive apparaît, cela confirme qu'une production plus élevée génère mécaniquement plus de défauts en valeur absolue (ce qui est normal). L'indicateur pertinent à surveiller est le **taux de défauts** (défauts / produits), non la valeur brute. Si une équipe présente une droite de tendance plus pentue que les autres, elle génère proportionnellement plus de défauts — ce qui mérite investigation.

**Explication technique :** `trendline='ols'` ajoute automatiquement une droite de régression par groupe (Ordinary Least Squares). Cela nécessite la bibliothèque `statsmodels` (installée avec plotly express par défaut).

## Tâche 5 — Bonus : temps d'arrêt vs défauts

**Choix du type :** Scatter plot avec marqueur de taille. L'objectif est de visualiser si les arrêts machine sont corrélés à un nombre de défauts élevé — une hypothèse opérationnelle fréquente (un redémarrage après arrêt génère souvent des pièces non conformes).

In [ ]:
fig4 = px.scatter(
    df,
    x='temps_arret_min',
    y='quantite_defauts',
    color='ligne',
    size='quantite_produite',              # taille du point = volume produit
    title="Relation entre durée d'arrêt et défauts qualité, par ligne",
    labels={
        'temps_arret_min': "Durée d'arrêt (minutes)",
        'quantite_defauts': 'Nombre de défauts',
        'ligne': 'Ligne',
        'quantite_produite': 'Volume produit'
    },
    opacity=0.7,
    trendline='ols',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig4.update_layout(
    plot_bgcolor='white'
)

fig4.show()

**Interprétation :** Ce graphique à trois dimensions encodées (X = arrêt, Y = défauts, taille = volume) permet de croiser plusieurs informations simultanément. Si la tendance est positive et prononcée, cela valide l'hypothèse que les arrêts machine dégradent la qualité au redémarrage. La recommandation opérationnelle serait alors d'optimiser les procédures de reprise après arrêt.